# Pinpoint · WordNet Edition (Juego en Consola / Colab)

Juego de adivinanza léxica basado en relaciones semánticas de **WordNet** (hiperónimos, hipónimos, sinónimos) y normalización morfológica con **SnowballStemmer**.

### Instrucciones:
1. Ejecuta cada celda en orden secuencial.
2. En la última celda, interactúa ingresando el idioma de las pistas (`spa`, `fra`, `eng`), el idioma de la respuesta y el número de rondas.

In [ ]:
# Celda 1: Descarga de recursos e inicialización de stemmers
import nltk
import unicodedata
import random as rd
from nltk.corpus import wordnet as wn
from nltk.stem.snowball import SnowballStemmer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

stemmer_fr = SnowballStemmer("french")
stemmer_en = SnowballStemmer("english")
stemmer_es = SnowballStemmer("spanish")

print("Dependencias y recursos cargados correctamente.")

In [ ]:
# Celda 2: Módulo de normalización y stemming (Lematización)
def normalize(texte: str) -> str:
    texte = texte.lower().strip()
    texte = unicodedata.normalize('NFD', texte)
    texte = ''.join(c for c in texte if unicodedata.category(c) != 'Mn')
    return texte

def stemmer_word(word: str, lang: str) -> str:
    original = normalize(word)
    if lang == "fra":
        stemmed = stemmer_fr.stem(original)
    elif lang == "spa":
        stemmed = stemmer_es.stem(original)
    elif lang == "eng":
        stemmed = stemmer_en.stem(original)
    else:
        return original

    # Regla de seguridad para tolerar plurales que el stemmer no reduzca
    if stemmed == original and original.endswith("s") and len(original) > 3:
        stemmed = original[:-1]
    return stemmed

def is_same_word(proposition: str, answer: str, lang: str) -> bool:
    return stemmer_word(proposition, lang) == stemmer_word(answer, lang)

In [ ]:
# Celda 3: Módulo de pistas léxicas y selección con WordNet
MOTS_ANGLAIS = [
    "guitar", "violin", "piano", "drum", "trumpet", "flute", "harp", "saxophone",
    "lawyer", "teacher", "doctor", "surgeon", "farmer", "engineer", "pilot", "nurse",
    "firefighter", "police_officer", "chef", "dentist", "architect", "plumber",
    "electrician", "mechanic", "astronaut", "scientist", "painter", "photographer",
    "elephant", "penguin", "kangaroo", "dolphin", "giraffe", "zebra", "tiger", "lion",
    "panda", "koala", "octopus", "eagle", "owl", "butterfly", "spider", "crocodile",
    "umbrella", "backpack", "camera", "telescope", "microscope", "compass", "map",
    "calculator", "keyboard", "mirror", "candle", "lantern", "basket", "ladder",
    "hospital", "airport", "library", "museum", "cathedral", "castle", "lighthouse",
    "stadium", "theater", "bakery", "pharmacy", "supermarket", "restaurant", "hotel",
    "bicycle", "helicopter", "sailboat", "submarine", "train", "airplane", "rocket",
    "motorcycle", "canoe", "tractor", "ambulance",
    "waterfall", "volcano", "mountain", "desert", "forest", "glacier", "island",
    "river", "cave", "cliff", "valley", "beach", "lake",
    "violinist", "carpenter", "blacksmith", "beekeeper",
]

MOTS_ESPAGNOL = [
    "guitarra", "violín", "piano", "tambor", "trompeta", "flauta", "arpa", "saxofón",
    "abogado", "profesor", "médico", "cirujano", "agricultor", "ingeniero", "piloto",
    "enfermera", "bombero", "policía", "cocinero", "dentista", "arquitecto", "fontanero",
    "electricista", "mecánico", "astronauta", "científico", "pintor", "fotógrafo",
    "elefante", "pingüino", "canguro", "delfín", "jirafa", "cebra", "tigre", "león",
    "panda", "koala", "pulpo", "águila", "búho", "mariposa", "araña", "cocodrilo",
    "paraguas", "mochila", "cámara", "telescopio", "microscopio", "brújula", "mapa",
    "calculadora", "teclado", "espejo", "vela", "linterna", "cesta", "escalera",
    "hospital", "aeropuerto", "biblioteca", "museo", "catedral", "castillo", "faro",
    "estadio", "teatro", "panadería", "farmacia", "supermercado", "restaurante", "hotel",
    "bicicleta", "helicóptero", "velero", "submarino", "tren", "avión", "cohete",
    "motocicleta", "canoa", "tractor", "ambulancia",
    "cascada", "volcán", "montaña", "desierto", "bosque", "glaciar", "isla",
    "río", "cueva", "acantilado", "valle", "playa", "lago",
    "violinista", "carpintero", "herrero", "apicultor",
]

MOTS_FRANCAIS = [
    "guitare", "violon", "piano", "tambour", "trompette", "flûte", "harpe", "saxophone",
    "avocat", "professeur", "médecin", "chirurgien", "agriculteur", "ingénieur", "pilote",
    "infirmière", "pompier", "policier", "cuisinier", "dentiste", "architecte", "plombier",
    "électricien", "mécanicien", "astronaute", "scientifique", "peintre", "photographe",
    "éléphant", "pingouin", "kangourou", "dauphin", "girafe", "zèbre", "tigre", "lion",
    "panda", "koala", "poulpe", "aigle", "hibou", "papillon", "araignée", "crocodile",
    "parapluie", "sac_à_dos", "appareil_photo", "télescope", "microscope", "boussole",
    "carte", "calculatrice", "clavier", "miroir", "bougie", "lanterne", "panier", "échelle",
    "hôpital", "aéroport", "bibliothèque", "musée", "cathédrale", "château", "phare",
    "stade", "théâtre", "boulangerie", "pharmacie", "supermarché", "restaurant", "hôtel",
    "vélo", "hélicoptère", "voilier", "sous-marin", "train", "avion", "fusée",
    "moto", "canoë", "tracteur", "ambulance",
    "cascade", "volcan", "montagne", "désert", "forêt", "glacier", "île",
    "rivière", "grotte", "falaise", "vallée", "plage", "lac",
    "violiniste", "charpentier", "forgeron", "apiculteur",
]

def nombre_seguro(synset, idioma):
    if idioma == "eng":
        lemas = synset.lemmas()
    else:
        lemas = synset.lemmas(lang=idioma)
    if not lemas:
        return None
    nombre = lemas[0].name().replace("_", " ")
    return nombre.split("|")[0]

def _banco(idioma):
    if idioma == "spa":
        return MOTS_ESPAGNOL
    elif idioma == "fra":
        return MOTS_FRANCAIS
    return MOTS_ANGLAIS

def obtener_synset_por_indice(indice, idioma):
    palabra_en = MOTS_ANGLAIS[indice]
    synsets_en = wn.synsets(palabra_en, pos=wn.NOUN)
    if not synsets_en:
        return None
    if idioma == "eng":
        return synsets_en[0]

    palabra_cible = _banco(idioma)[indice]
    synsets_cible = set(wn.synsets(palabra_cible, lang=idioma, pos=wn.NOUN))
    if not synsets_cible:
        return None

    for s in synsets_en:
        if s in synsets_cible:
            return s
    return next(iter(synsets_cible))

def crear_palabras(idioma_i, max_intentos=500):
    n = len(MOTS_ANGLAIS)
    for _ in range(max_intentos):
        indice = rd.randrange(n)
        s = obtener_synset_por_indice(indice, idioma_i)
        if s is None:
            continue
        if (len(s.hypernyms()) > 0
                and len(s.hyponyms()) > 1
                and nombre_seguro(s, idioma_i) is not None):
            return indice
    raise RuntimeError("No se encontró una palabra válida en el banco.")

def obtener_indicios(indice, idioma_i):
    synset = obtener_synset_por_indice(indice, idioma_i)
    palabra = _banco(idioma_i)[indice]
    indicios = []
    origen = []

    hyperonymos = synset.hypernyms()
    hyponyms = synset.hyponyms()

    # 1. Hiperónimos
    candidatos_hiper = list(hyperonymos)
    if hyperonymos:
        candidatos_hiper += hyperonymos[0].hypernyms()

    for h in candidatos_hiper:
        if len(indicios) >= 2:
            break
        nombre = nombre_seguro(h, idioma_i)
        if nombre and nombre.lower() != palabra.replace("_", " ").lower() and nombre not in indicios:
            indicios.append(nombre)
            origen.append("hiperónimo")

    # 2. Hipónimos
    rd.shuffle(hyponyms)
    for h in hyponyms:
        if len(indicios) >= 4:
            break
        nombre = nombre_seguro(h, idioma_i)
        if nombre and nombre.lower() != palabra.replace("_", " ").lower() and nombre not in indicios:
            indicios.append(nombre)
            origen.append("hipónimo")

    # 3. Sinónimos
    if idioma_i == "eng":
        lemas = synset.lemmas()
    else:
        lemas = synset.lemmas(lang=idioma_i)

    sinonimos = [l.name().replace("_", " ").split("|")[0] for l in lemas]
    sinonimos = [s for s in sinonimos if s.lower() != palabra.replace("_", " ").lower()]
    rd.shuffle(sinonimos)
    for s in sinonimos:
        if len(indicios) >= 5:
            break
        if s not in indicios:
            indicios.append(s)
            origen.append("sinónimo")

    # 4. Reserva: hipónimos de hipónimos
    if len(indicios) < 5:
        for h in hyponyms:
            for nieto in h.hyponyms():
                nombre = nombre_seguro(nieto, idioma_i)
                if nombre and nombre.lower() != palabra.replace("_", " ").lower() and nombre not in indicios:
                    indicios.append(nombre)
                    origen.append("hipónimo")
                if len(indicios) >= 5:
                    break
            if len(indicios) >= 5:
                break

    return indicios[:5], origen[:5]

def generar_partida(idioma_i, idioma_preg, max_intentos=100):
    for _ in range(max_intentos):
        indice = crear_palabras(idioma_i)
        indicios, origen = obtener_indicios(indice, idioma_i)
        if len(indicios) == 5:
            palabra_pista = _banco(idioma_i)[indice]
            palabra_respuesta = _banco(idioma_preg)[indice]
            return palabra_pista, indicios, origen, palabra_respuesta

    palabra_pista = _banco(idioma_i)[indice]
    palabra_respuesta = _banco(idioma_preg)[indice]
    return palabra_pista, indicios, origen, palabra_respuesta

In [ ]:
# Celda 4: Lógica de la partida e interfaz de usuario por consola
IDIOMAS_VALIDOS = ("eng", "fra", "spa")
NOMBRES_IDIOMA = {"eng": "inglés", "fra": "francés", "spa": "español"}

class Game:
    def __init__(self, round_count, langage_pistas, langage_respuesta, generar_partida_fn):
        self.langage_pistas = langage_pistas
        self.langage_respuesta = langage_respuesta
        self.rounds = []
        while len(self.rounds) < round_count:
            palabra_pista, indicios, origen, palabra_respuesta = generar_partida_fn(
                langage_pistas, langage_respuesta
            )
            if palabra_pista and indicios and palabra_respuesta:
                self.rounds.append({
                    "palabra_pista": palabra_pista,
                    "indicios": indicios,
                    "origen": origen,
                    "palabra_respuesta": palabra_respuesta,
                })
        self.r = 0
        self.clues = 0
        self.score = 0
        self.round_finished = False
        self.n_round = round_count

    def is_last(self):
        return self.r == (self.n_round - 1)

    def answer(self, text):
        if self.round_finished:
            return "Round_finished"
        word_normalised = normalize(text)
        if not word_normalised:
            return "Empty"
        respuesta_esperada = self.rounds[self.r]["palabra_respuesta"]
        if is_same_word(text, respuesta_esperada, self.langage_respuesta):
            self.score += 6 - self.clues
            self.round_finished = True
            return "words_found"
        if self.clues < 4:
            self.clues += 1
            return "wrong"
        self.round_finished = True
        return "out_of_clues"

    def next_round(self):
        if not self.round_finished or self.is_last():
            return False
        self.r += 1
        self.clues = 0
        self.round_finished = False
        return True

def pedir_idioma(mensaje):
    idioma = ""
    while idioma not in IDIOMAS_VALIDOS:
        idioma = input(mensaje).strip().lower()
        if idioma not in IDIOMAS_VALIDOS:
            print("Idioma no válido. Ingrese 'eng', 'fra' o 'spa'.")
    return idioma

def pedir_numero_rondas():
    while True:
        texto = input("¿Cuántas rondas deseas jugar? (ej: 3): ").strip()
        if texto.isdigit() and int(texto) > 0:
            return int(texto)
        print("Por favor, ingrese un número entero positivo.")

def play_console():
    print("=== Pinpoint · WordNet Edition ===\n")
    lang_pistas = pedir_idioma("Idioma de las pistas (eng, fra, spa): ")
    lang_respuesta = pedir_idioma("Idioma de la respuesta esperada (eng, fra, spa): ")
    n_round = pedir_numero_rondas()

    print(f"\nPistas en {NOMBRES_IDIOMA[lang_pistas]}, respuestas en {NOMBRES_IDIOMA[lang_respuesta]}.")
    print("Generando partida...\n")
    game = Game(n_round, lang_pistas, lang_respuesta, generar_partida)

    while True:
        print(f"=== Ronda {game.r + 1}/{game.n_round} | Puntaje: {game.score} ===")
        while not game.round_finished:
            clue = game.rounds[game.r]["indicios"][game.clues]
            origen = game.rounds[game.r]["origen"][game.clues]
            print(f"Pista {game.clues + 1} [{origen}]: {clue}")
            text = input("  Tu respuesta (o 'exit' para salir): ").strip()
            if text.lower() == "exit":
                print(f"\nPartida cancelada. Puntaje final: {game.score}")
                return
            resultado = game.answer(text)
            if resultado == "Empty":
                print("  Escribe una respuesta.")
            elif resultado == "wrong":
                print("  Incorrecto, aquí tienes otra pista...")
            elif resultado == "words_found":
                print(f"  ¡Correcto! La respuesta era '{game.rounds[game.r]['palabra_respuesta']}'. Puntaje: {game.score}\n")
            elif resultado == "out_of_clues":
                print(f"  Sin pistas. La respuesta correcta era '{game.rounds[game.r]['palabra_respuesta']}'.\n")
        if not game.next_round():
            break
    print(f"=== Fin de la partida. Puntaje final: {game.score} ===")

# Iniciar juego
play_console()